# 03 — Model Comparison
Train all 5 classifiers on ResNet50 embeddings and compare performance comprehensively.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, ConfusionMatrixDisplay
from src.embeddings import load_embeddings
from src.models import get_classifier, train_classifier, predict_classifier
from src.evaluation import evaluate_classifier

X_train, y_train = load_embeddings('train')
X_val,   y_val   = load_embeddings('val')
X_test,  y_test  = load_embeddings('test')
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

## Train & Evaluate All 5 Models

In [ ]:
MODEL_NAMES = ['svm', 'logistic_regression', 'random_forest', 'adaboost', 'lda']
results = []
trained_models = {}

for name in MODEL_NAMES:
    print(f"Training {name}...", end=' ')
    clf = get_classifier(name, random_state=42)
    clf = train_classifier(clf, X_train, y_train)
    preds, probs = predict_classifier(clf, X_test)
    metrics = evaluate_classifier(y_test, preds, probs, is_multiclass=False)
    results.append({
        'Model': name.replace('_',' ').title(),
        'Test Accuracy': round(metrics.get('Accuracy', 0), 4),
        'Test F1':       round(metrics.get('F1-Score', 0), 4),
        'Test Precision':round(metrics.get('Precision', 0), 4),
        'Test Recall':   round(metrics.get('Recall', 0), 4),
        'Test AUC':      round(float(metrics.get('AUC-ROC', 0)), 4),
    })
    trained_models[name] = (clf, preds, probs)
    print("done")

df_results = pd.DataFrame(results).set_index('Model')
df_results.sort_values('Test AUC', ascending=False)

## Performance Comparison Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(df_results))
w = 0.18
metrics_to_plot = ['Test Accuracy','Test F1','Test Precision','Test Recall','Test AUC']
colors = ['#3498db','#e67e22','#2ecc71','#9b59b6','#e74c3c']

for i, (col, color) in enumerate(zip(metrics_to_plot, colors)):
    bars = ax.bar(x + i*w, df_results[col], w, label=col, color=color, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + 2*w)
ax.set_xticklabels(df_results.index, rotation=10)
ax.set_ylim(0.6, 1.05); ax.set_ylabel('Score')
ax.set_title('All 5 Models — Performance on Real ISIC Test Set', fontsize=13, fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## ROC Curves — All Models Overlaid

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
colors = ['#3498db','#e67e22','#2ecc71','#9b59b6','#e74c3c']

for (name, (clf, preds, probs)), color in zip(trained_models.items(), colors):
    if probs is not None:
        fpr, tpr, _ = roc_curve(y_test, probs)
        roc_auc = auc(fpr, tpr)
        label = f"{name.replace('_',' ').title()} (AUC={roc_auc:.3f})"
        ax.plot(fpr, tpr, color=color, lw=2, label=label)

ax.plot([0,1],[0,1],'k--', lw=1, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Classifiers', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Confusion Matrices — Best 3 Models

In [ ]:
top3 = df_results.sort_values('Test AUC', ascending=False).head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, model_key in zip(axes, [n.lower().replace(' ','_') for n in top3]):
    clf, preds, _ = trained_models[model_key]
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds, display_labels=['Benign','Malignant'],
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(model_key.replace('_',' ').title(), fontweight='bold')
plt.suptitle('Confusion Matrices — Top 3 Models', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()